In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.db import test_connection

test_connection()

('neondb', 'ecom', 'PostgreSQL 18.7 (84da7d8) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit')

In [1]:
import sys
print(sys.version)
print(sys.executable)

3.13.18 (main, Oct 25 2026, 18:31:23) [Clang 21.0.0 (clang-2100.3.34.2)]
/Users/neharika/Python_Projects/rfm-customer-segmentation/.venv/bin/python


In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import pandas as pd
from sqlalchemy import text
from src.db import engine

query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'ecom'
  AND table_type = 'BASE TABLE'
ORDER BY table_name;
"""

with engine.connect() as connection:
    tables = pd.read_sql(text(query), connection)

tables

,table_name
0,addresses
1,attribution_campaigns
2,attribution_touches
3,brands
4,categories
5,collection_products
6,collections
7,consents
8,coupons
9,customer_addresses


In [4]:
query = """
SELECT
    table_name,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'ecom'
  AND table_name IN (
      'customers',
      'orders',
      'order_items',
      'customer_segments',
      'refunds'
  )
ORDER BY table_name, ordinal_position;
"""

with engine.connect() as connection:
    schema_df = pd.read_sql(text(query), connection)

schema_df

,table_name,column_name,data_type
0,customer_segments,segment_id,bigint
1,customer_segments,segment_name,text
2,customers,customer_id,bigint
3,customers,created_at,timestamp with time zone
4,customers,first_name,text
5,customers,last_name,text
6,customers,dob,date
7,customers,gender,text
8,customers,primary_email,text
9,customers,primary_phone,text


In [5]:
query = """
SELECT 'customers' AS table_name,
       COUNT(*) AS row_count,
       MIN(created_at) AS first_record,
       MAX(created_at) AS last_record
FROM ecom.customers

UNION ALL

SELECT 'orders',
       COUNT(*),
       MIN(created_at),
       MAX(created_at)
FROM ecom.orders

UNION ALL

SELECT 'order_items',
       COUNT(*),
       NULL,
       NULL
FROM ecom.order_items

UNION ALL

SELECT 'refunds',
       COUNT(*),
       MIN(created_at),
       MAX(created_at)
FROM ecom.refunds;
"""

with engine.connect() as connection:
    overview = pd.read_sql(text(query), connection)

overview

,table_name,row_count,first_record,last_record
0,refunds,260,2026-03-20 20:36:16.082982+00:00,2026-06-23 08:44:06.611335+00:00
1,customers,10000,2026-03-16 11:03:35.000000+00:00,2026-06-14 10:57:52.000000+00:00
2,orders,40000,2026-03-16 00:13:57.000000+00:00,2026-06-14 23:28:40.000000+00:00
3,order_items,81806,NaT,NaT


In [6]:
query = """
SELECT
    status,
    COUNT(*) AS order_count,
    COUNT(DISTINCT customer_id) AS unique_customers,
    MIN(created_at) AS first_order,
    MAX(created_at) AS last_order
FROM ecom.orders
GROUP BY status
ORDER BY order_count DESC;
"""

with engine.connect() as connection:
    status_summary = pd.read_sql(text(query), connection)

status_summary

,status,order_count,unique_customers,first_order,last_order
0,delivered,19779,6032,2026-03-16 00:13:57+00:00,2026-06-14 23:28:40+00:00
1,shipped,7715,3641,2026-03-16 01:10:06+00:00,2026-06-14 22:29:50+00:00
2,paid,3946,2359,2026-03-16 00:53:05+00:00,2026-06-14 16:00:09+00:00
3,packed,3887,2341,2026-03-16 03:52:19+00:00,2026-06-14 16:52:43+00:00
4,cancelled,2178,1493,2026-03-16 04:44:46+00:00,2026-06-14 16:36:40+00:00
5,placed,1897,1367,2026-03-16 07:36:34+00:00,2026-06-14 15:12:44+00:00
6,SHIPPED,248,229,2026-03-16 11:21:14+00:00,2026-06-14 15:14:00+00:00
7,DELIVERED,200,183,2026-03-17 14:15:12+00:00,2026-06-13 17:09:24+00:00
8,Shipped,150,146,2026-03-16 02:09:22+00:00,2026-06-13 16:58:48+00:00


In [7]:
query = """
SELECT
    LOWER(TRIM(status)) AS clean_status,
    COUNT(*) AS order_count,
    COUNT(DISTINCT customer_id) AS unique_customers,
    MIN(created_at) AS first_order,
    MAX(created_at) AS last_order
FROM ecom.orders
GROUP BY LOWER(TRIM(status))
ORDER BY order_count DESC;
"""

with engine.connect() as connection:
    clean_status_summary = pd.read_sql(text(query), connection)

clean_status_summary

,clean_status,order_count,unique_customers,first_order,last_order
0,delivered,19979,6052,2026-03-16 00:13:57+00:00,2026-06-14 23:28:40+00:00
1,shipped,8113,3743,2026-03-16 01:10:06+00:00,2026-06-14 22:29:50+00:00
2,paid,3946,2359,2026-03-16 00:53:05+00:00,2026-06-14 16:00:09+00:00
3,packed,3887,2341,2026-03-16 03:52:19+00:00,2026-06-14 16:52:43+00:00
4,cancelled,2178,1493,2026-03-16 04:44:46+00:00,2026-06-14 16:36:40+00:00
5,placed,1897,1367,2026-03-16 07:36:34+00:00,2026-06-14 15:12:44+00:00


In [8]:
query = """
SELECT *
FROM ecom.refunds
LIMIT 10;
"""

with engine.connect() as connection:
    refunds_sample = pd.read_sql(text(query), connection)

refunds_sample

,refund_id,order_id,created_at,amount,reason,status
0,3,21066,2026-05-09 22:16:41.274820+00:00,7251.23,return_refund,succeeded
1,4,28132,2026-05-31 15:48:33.632061+00:00,9848.28,return_refund,succeeded
2,6,11815,2026-04-19 18:20:09.221633+00:00,4244.46,customer_request,succeeded
3,7,23663,2026-05-12 08:54:11.333579+00:00,938.58,customer_request,succeeded
4,8,8528,2026-04-13 18:40:47.517029+00:00,11394.44,duplicate_charge,succeeded
5,10,37865,2026-06-12 09:54:21.699739+00:00,6019.00,return_refund,succeeded
6,11,26644,2026-05-20 06:36:03.467278+00:00,7075.28,return_refund,succeeded
7,12,2756,2026-03-29 13:52:22.368970+00:00,13169.34,return_refund,succeeded
8,13,7163,2026-04-07 03:45:21.389750+00:00,2948.82,return_refund,succeeded
9,14,35485,2026-04-12 20:47:19.710177+00:00,3773.64,return_refund,initiated


In [9]:
query = """
SELECT
    status,
    COUNT(*) AS refund_count,
    COUNT(DISTINCT order_id) AS unique_orders,
    SUM(amount) AS total_refund_amount,
    AVG(amount) AS average_refund_amount
FROM ecom.refunds
GROUP BY status
ORDER BY refund_count DESC;
"""

with engine.connect() as connection:
    refund_summary = pd.read_sql(text(query), connection)

refund_summary

,status,refund_count,unique_orders,total_refund_amount,average_refund_amount
0,succeeded,227,227,1058291.82,4662.078502
1,initiated,20,20,92962.95,4648.147500
2,failed,13,13,46380.56,3567.735385


In [10]:
query = """
SELECT
    LOWER(TRIM(o.status)) AS order_status,
    r.status AS refund_status,
    COUNT(*) AS refund_count,
    SUM(r.amount) AS refund_amount
FROM ecom.refunds r
JOIN ecom.orders o
    ON r.order_id = o.order_id
GROUP BY
    LOWER(TRIM(o.status)),
    r.status
ORDER BY
    order_status,
    refund_status;
"""

with engine.connect() as connection:
    refund_order_analysis = pd.read_sql(text(query), connection)

refund_order_analysis

,order_status,refund_status,refund_count,refund_amount
0,delivered,failed,13,46380.56
1,delivered,initiated,20,92962.95
2,delivered,succeeded,227,1058291.82


In [13]:
query = """
WITH delivered_orders AS (
    SELECT
        order_id,
        customer_id,
        total
    FROM ecom.orders
    WHERE LOWER(TRIM(status)) = 'delivered'
),

successful_refunds AS (
    SELECT
        order_id,
        SUM(amount) AS refund_amount
    FROM ecom.refunds
    WHERE status = 'succeeded'
    GROUP BY order_id
)

SELECT
    COUNT(*) AS delivered_orders,

    ROUND(SUM(o.total), 2) AS gross_order_value,

    ROUND(COALESCE(SUM(r.refund_amount), 0), 2)
        AS successful_refunds,

    ROUND(
    SUM(o.total) - COALESCE(SUM(r.refund_amount), 0),
    2
    )
        AS net_order_value

FROM delivered_orders o
LEFT JOIN successful_refunds r
    ON o.order_id = r.order_id;
"""

with engine.connect() as connection:
    revenue_summary = pd.read_sql(text(query), connection)

pd.options.display.float_format = '{:.2f}'.format

revenue_summary

,delivered_orders,gross_order_value,successful_refunds,net_order_value
0,19979,150418237.66,1058291.82,149359945.84


In [14]:

query = """
WITH customer_activity AS (
    SELECT
        c.customer_id,

        COUNT(o.order_id) AS total_orders,

        COUNT(o.order_id) FILTER (
            WHERE LOWER(TRIM(o.status)) = 'delivered'
        ) AS delivered_orders,

        COALESCE(
            SUM(o.total) FILTER (
                WHERE LOWER(TRIM(o.status)) = 'delivered'
            ), 0
        ) AS delivered_spend

    FROM ecom.customers c

    LEFT JOIN ecom.orders o
        ON c.customer_id = o.customer_id

    GROUP BY c.customer_id
)

SELECT
    COUNT(*) AS total_customers,

    COUNT(*) FILTER (
        WHERE total_orders > 0
    ) AS customers_with_orders,

    COUNT(*) FILTER (
        WHERE total_orders = 0
    ) AS customers_without_orders,

    COUNT(*) FILTER (
        WHERE delivered_orders > 0
    ) AS customers_with_delivered_orders,

    ROUND(
        AVG(delivered_orders) FILTER (
            WHERE delivered_orders > 0
        ), 2
    ) AS avg_delivered_orders_per_active_customer,

    ROUND(
        AVG(delivered_spend) FILTER (
            WHERE delivered_orders > 0
        ), 2
    ) AS avg_delivered_spend_per_active_customer

FROM customer_activity;
"""

with engine.connect() as connection:
    customer_coverage = pd.read_sql(
        text(query), connection
    )

customer_coverage


,total_customers,customers_with_orders,customers_without_orders,customers_with_delivered_orders,avg_delivered_orders_per_active_customer,avg_delivered_spend_per_active_customer
0,10000,8800,1200,6052,3.30,24854.30


In [15]:

query = """
WITH customer_spend AS (
    SELECT
        customer_id,
        SUM(total) AS delivered_spend
    FROM ecom.orders
    WHERE LOWER(TRIM(status)) = 'delivered'
    GROUP BY customer_id
)

SELECT
    COUNT(*) AS active_customers,

    ROUND(AVG(delivered_spend), 2) AS avg_spend,

    ROUND(
        percentile_cont(0.50)
        WITHIN GROUP (ORDER BY delivered_spend)::numeric, 2
    ) AS median_spend,

    ROUND(
        percentile_cont(0.75)
        WITHIN GROUP (ORDER BY delivered_spend)::numeric, 2
    ) AS p75_spend,

    ROUND(
        percentile_cont(0.90)
        WITHIN GROUP (ORDER BY delivered_spend)::numeric, 2
    ) AS p90_spend,

    ROUND(
        percentile_cont(0.95)
        WITHIN GROUP (ORDER BY delivered_spend)::numeric, 2
    ) AS p95_spend,

    ROUND(MIN(delivered_spend), 2) AS min_spend,

    ROUND(MAX(delivered_spend), 2) AS max_spend

FROM customer_spend;
"""

with engine.connect() as connection:
    spending_distribution = pd.read_sql(
        text(query), connection
    )

spending_distribution


,active_customers,avg_spend,median_spend,p75_spend,p90_spend,p95_spend,min_spend,max_spend
0,6052,24854.30,11441.87,28363.07,55244.83,82215.83,175.82,795745.26


In [ ]:

query = """
WITH customer_spend AS (
    SELECT
        customer_id,
        SUM(total) AS delivered_spend
    FROM ecom.orders
    WHERE LOWER(TRIM(status)) = 'delivered'
    GROUP BY customer_id
),

ranked_customers AS (
    SELECT
        customer_id,
        delivered_spend,

        ROW_NUMBER() OVER (
            ORDER BY delivered_spend DESC
        ) AS customer_rank,

        COUNT(*) OVER () AS total_customers,

        SUM(delivered_spend) OVER () AS total_revenue

    FROM customer_spend
)

SELECT
    MAX(total_customers) AS total_customers,
    ROUND(MAX(total_revenue), 2) AS total_revenue,
    ROUND(
        100.0 * SUM(delivered_spend)
        FILTER (
            WHERE customer_rank <= CEIL(total_customers * 0.10)
        ) / MAX(total_revenue), 2
    ) AS top_10_pct_revenue_share,

    ROUND(
        100.0 * SUM(delivered_spend)
        FILTER (
            WHERE customer_rank <= CEIL(total_customers * 0.20)
        ) / MAX(total_revenue), 2
    ) AS top_20_pct_revenue_share,

    ROUND(
        100.0 * SUM(delivered_spend)
        FILTER (
            WHERE customer_rank <= CEIL(total_customers * 0.50)
        ) / MAX(total_revenue), 2
    ) AS top_50_pct_revenue_share

FROM ranked_customers;
"""

with engine.connect() as connection:
    revenue_concentration = pd.read_sql(
        text(query), connection
    )

revenue_concentration


,total_customers,total_revenue,top_10_pct_revenue_share,top_20_pct_revenue_share,top_50_pct_revenue_share
0,6052,150418237.66,47.97,65.21,90.45


In [17]:

query = """
WITH customer_activity AS (
    SELECT
        customer_id,
        COUNT(order_id) AS delivered_orders,
        SUM(total) AS delivered_spend,
        AVG(total) AS avg_order_value

    FROM ecom.orders
    WHERE LOWER(TRIM(status)) = 'delivered'
    GROUP BY customer_id
),

ranked_customers AS (
    SELECT
        customer_id,
        delivered_orders,
        delivered_spend,
        avg_order_value,

        ROW_NUMBER() OVER (
            ORDER BY delivered_spend DESC
        ) AS customer_rank,

        COUNT(*) OVER () AS total_customers

    FROM customer_activity
),

customer_groups AS (
    SELECT
        CASE
            WHEN customer_rank <= CEIL(total_customers * 0.10)
            THEN 'Top 10%'
            ELSE 'Remaining 90%'
        END AS customer_group,

        delivered_orders,
        delivered_spend,
        avg_order_value

    FROM ranked_customers
)

SELECT
    customer_group,

    COUNT(*) AS customer_count,

    ROUND(AVG(delivered_orders), 2)
        AS avg_orders_per_customer,

    ROUND(AVG(delivered_spend), 2)
        AS avg_spend_per_customer,

    ROUND(AVG(avg_order_value), 2)
        AS avg_order_value,

    ROUND(SUM(delivered_spend), 2)
        AS total_revenue

FROM customer_groups
GROUP BY customer_group
ORDER BY
    CASE customer_group
        WHEN 'Top 10%' THEN 1
        ELSE 2
    END;
"""

with engine.connect() as connection:
    frequency_analysis = pd.read_sql(
        text(query), connection
    )

frequency_analysis


,customer_group,customer_count,avg_orders_per_customer,avg_spend_per_customer,avg_order_value,total_revenue
0,Top 10%,606,13.08,119071.23,13582.77,72157166.86
1,Remaining 90%,5446,2.21,14370.38,6939.61,78261070.80


In [18]:
query = """
SELECT
    MAX(created_at) AS latest_order_date,
    MAX(created_at) FILTER (
        WHERE LOWER(TRIM(status)) = 'delivered'
    ) AS latest_delivered_order_date
FROM ecom.orders;
"""

with engine.connect() as connection:
    reference_dates = pd.read_sql(text(query), connection)

reference_dates

,latest_order_date,latest_delivered_order_date
0,2026-06-14 23:28:40+00:00,2026-06-14 23:28:40+00:00


In [19]:
query = """
SELECT
    customer_id,
    MAX(created_at) AS last_purchase_date,
    DATE '2026-06-15' - MAX(created_at)::date AS recency_days
FROM ecom.orders
WHERE LOWER(TRIM(status)) = 'delivered'
GROUP BY customer_id
ORDER BY recency_days;
"""

with engine.connect() as connection:
    recency_df = pd.read_sql(text(query), connection)

recency_df.head(10)

,customer_id,last_purchase_date,recency_days
0,2152,2026-06-14 07:37:25+00:00,1
1,7505,2026-06-14 13:51:51+00:00,1
2,6190,2026-06-14 05:05:18+00:00,1
3,5490,2026-06-14 23:28:40+00:00,1
4,3954,2026-06-14 05:21:43+00:00,1
5,7000,2026-06-14 08:51:12+00:00,1
6,394,2026-06-14 03:55:11+00:00,1
7,6080,2026-06-14 15:30:28+00:00,1
8,5152,2026-06-14 19:10:11+00:00,1
9,8546,2026-06-14 07:12:30+00:00,1


In [20]:
query = """
WITH customer_recency AS (
    SELECT
        customer_id,
        DATE '2026-06-15' - MAX(created_at)::date AS recency_days
    FROM ecom.orders
    WHERE LOWER(TRIM(status)) = 'delivered'
    GROUP BY customer_id
)
SELECT
    COUNT(*) AS customers,
    ROUND(AVG(recency_days), 2) AS avg_recency,
    PERCENTILE_CONT(0.50) WITHIN GROUP (
        ORDER BY recency_days
    ) AS median_recency,
    MIN(recency_days) AS min_recency,
    MAX(recency_days) AS max_recency
FROM customer_recency;
"""

with engine.connect() as connection:
    recency_summary = pd.read_sql(text(query), connection)

recency_summary

,customers,avg_recency,median_recency,min_recency,max_recency
0,6052,41.06,39.00,1,91


In [ ]:
query = """
WITH customer_recency AS (
    SELECT
        customer_id,
        DATE '2026-06-15' - MAX(created_at)::date AS recency_days
    FROM ecom.orders
    WHERE LOWER(TRIM(status)) = 'delivered'
    GROUP BY customer_id
)
SELECT
    CASE
        WHEN recency_days <= 30 THEN '1-30 days'
        WHEN recency_days <= 60 THEN '31-60 days'
        ELSE '61-91 days'
    END AS recency_band,
    COUNT(*) AS customer_count,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS customer_percentage
FROM customer_recency
GROUP BY 1
ORDER BY MIN(recency_days);
"""

#--COUNT(*) counts customers in each band.
#--SUM(...) OVER () adds those band counts together, while retaining each band as a separate row.
#--COUNT(*) counts customers in each Recency band.
#--SUM(...) adds those individual band counts.
#--OVER () makes that sum available across all groups without collapsing them into one row.

with engine.connect() as connection:
    recency_bands = pd.read_sql(text(query), connection)

recency_bands

,recency_band,customer_count,customer_percentage
0,1-30 days,2532,41.84
1,31-60 days,1753,28.97
2,61-91 days,1767,29.20
